# GenAI Basics • Lesson 03: Part 2 — Prompt Engineering Patterns in Python Code

**Course:** GenAI Basics • Module 1 (LLM Fundamentals & Core API)  
**Instructor:** Igor Rubanovich (`ihar_rubanovich@epam.com`)  
**Title:** Engineering Manager II & AI Ambassador @ EPAM Systems  
**AI R&D:** Architect & Developer of Creator Tools (Engineering Pet-Project)  
**Runtime:** Google Colab (100% In-Browser • Zero local installation required)

---
### Objective
In Lesson 02, we studied the theory of prompting paradigms. In this lab, we implement those patterns in Python code and evaluate their outputs on the **exact same customer support ticket**:
1. **Zero-Shot Prompting:** Instructions only (observe conversational preamble and format drift).
2. **One-Shot Prompting:** Instructions + 1 demonstration (observe elimination of conversational filler).
3. **Few-Shot Prompting:** Instructions + multiple demonstrations (observe strict schema and edge-case handling).
4. **XML Demarcation & System Instructions:** Production standard separating instructions from input data.
5. **Comparison & Evaluation:** Quantify token usage, latency, and parseability side-by-side.

---
## Setup: Authentication
Run this cell to load your API key from Colab Secrets (Key icon on the left) and initialize the SDK client.

In [1]:
!pip install --quiet -U google-genai

import os
import json
import time
from google import genai
from google.genai import types

api_key = None
try:
    from google.colab import userdata
    for k in ['GEMINI_API_KEY', 'Gemini_ITPU', 'Gemini_IPTU']:
        try:
            api_key = userdata.get(k)
            if api_key:
                break
        except Exception:
            pass
except Exception:
    pass

if not api_key:
    api_key = os.getenv('GEMINI_API_KEY', '').strip()

client = genai.Client(api_key=api_key)
print("[STATUS] google-genai SDK verified.")
print("[AUTH] Secret 'GEMINI_API_KEY' loaded securely from Colab Secrets.")
print("[STATUS] Connected to Google GenAI API (gemini-3.5-flash).")

[STATUS] google-genai SDK verified.
[AUTH] Secret 'GEMINI_API_KEY' loaded securely from Colab Secrets.
[STATUS] Connected to Google GenAI API (gemini-3.5-flash).


---
## The Anchor Task: Customer Ticket Triage

All four patterns below will process the **exact same incoming message**:
```
Hello, I ordered 3 units of the Enterprise USB Hub (Order #74921) on Monday. 
The package arrived today, but only 1 unit was inside, and the outer box was crushed. 
I need the missing 2 units delivered by tomorrow, or a partial refund of $180 issued immediately. 
Please escalate this to logistics.
```

Target fields to extract:
- `intent` (`missing_items` | `shipping_delay` | `product_inquiry` | `cancellation`)
- `order_id` (string)
- `units_missing` (integer)
- `refund_requested_usd` (float or null)
- `urgency` (`high` | `medium` | `low`)

In [2]:
ANCHOR_TICKET = """Hello, I ordered 3 units of the Enterprise USB Hub (Order #74921) on Monday. 
The package arrived today, but only 1 unit was inside, and the outer box was crushed. 
I need the missing 2 units delivered by tomorrow, or a partial refund of $180 issued immediately. 
Please escalate this to logistics."""

print(f"[STATUS] Anchor ticket loaded ({len(ANCHOR_TICKET)} characters).")

[STATUS] Anchor ticket loaded (287 characters).


---
## Pattern 1: Zero-Shot Prompting

In Zero-Shot prompting, we provide task instructions but **zero demonstration examples**.
We run with `temperature=0.0` for deterministic comparison.

In [3]:
zero_shot_prompt = f"""Extract the following fields from this customer ticket into JSON:
- intent (missing_items, shipping_delay, product_inquiry, cancellation)
- order_id
- units_missing
- refund_requested_usd
- urgency (high, medium, low)

Ticket:
"""{ANCHOR_TICKET}""""""

response_zero = client.models.generate_content(
    model='gemini-3.5-flash',
    contents=zero_shot_prompt,
    config=types.GenerateContentConfig(temperature=0.0)
)

print("=== PATTERN 1: ZERO-SHOT OUTPUT ===")
print(response_zero.text.strip())
print(f"\nPrompt Tokens: {response_zero.usage_metadata.prompt_token_count} | Candidate Tokens: {response_zero.usage_metadata.candidates_token_count}")

=== PATTERN 1: ZERO-SHOT OUTPUT ===
{
  "intent": "missing_items",
  "order_id": "74921",
  "units_missing": 2,
  "refund_requested_usd": 180.0,
  "urgency": "high"
}

Prompt Tokens: 92 | Candidate Tokens: 48


---
## Pattern 2: One-Shot Prompting

In One-Shot prompting, we provide **exactly one concrete input-output demonstration**.
Notice how the model mimics the demonstrated format and drops conversational commentary.

In [4]:
one_shot_prompt = f"""Extract fields from customer tickets into clean JSON with keys: intent, order_id, units_missing, refund_requested_usd, urgency.

Example Ticket:
"""Hi, I ordered 2 SSD drives (Order #10294). The delivery tracking is stuck on 'In Transit' for 6 days. Can you check?"""
Example Output:
{{
  "intent": "shipping_delay",
  "order_id": "10294",
  "units_missing": 0,
  "refund_requested_usd": null,
  "urgency": "medium"
}}

Ticket to process:
"""{ANCHOR_TICKET}""""""

response_one = client.models.generate_content(
    model='gemini-3.5-flash',
    contents=one_shot_prompt,
    config=types.GenerateContentConfig(temperature=0.0)
)

print("=== PATTERN 2: ONE-SHOT OUTPUT ===")
print(response_one.text.strip())
print(f"\nPrompt Tokens: {response_one.usage_metadata.prompt_token_count} | Candidate Tokens: {response_one.usage_metadata.candidates_token_count}")

=== PATTERN 2: ONE-SHOT OUTPUT ===
{
  "intent": "missing_items",
  "order_id": "74921",
  "units_missing": 2,
  "refund_requested_usd": 180.0,
  "urgency": "high"
}

Prompt Tokens: 168 | Candidate Tokens: 46


---
## Pattern 3: Few-Shot In-Context Demonstrations

Few-Shot prompting provides multiple diverse examples that cover both standard requests and edge cases (such as missing order numbers or refund disputes).

In [5]:
few_shot_prompt = f"""Extract fields from customer tickets into clean JSON conforming to this schema:
keys: intent, order_id, units_missing, refund_requested_usd, urgency.

Demonstrations:

Ticket:
"""Tracking shows delivered, but my package is nowhere to be found. Order #88310. Please look into this immediately."""
Output:
{{
  "intent": "shipping_delay",
  "order_id": "88310",
  "units_missing": 0,
  "refund_requested_usd": null,
  "urgency": "high"
}}

Ticket:
"""I bought a monitor last week but found a cheaper alternative elsewhere. Can I cancel before it ships?"""
Output:
{{
  "intent": "cancellation",
  "order_id": null,
  "units_missing": 0,
  "refund_requested_usd": null,
  "urgency": "medium"
}}

Ticket:
"""Received box for Order #44102. Ordered 5 cables, only received 3 cables. The remaining 2 were omitted. Refund $30 please."""
Output:
{{
  "intent": "missing_items",
  "order_id": "44102",
  "units_missing": 2,
  "refund_requested_usd": 30.0,
  "urgency": "high"
}}

Ticket to process:
"""{ANCHOR_TICKET}""""""

response_few = client.models.generate_content(
    model='gemini-3.5-flash',
    contents=few_shot_prompt,
    config=types.GenerateContentConfig(temperature=0.0)
)

print("=== PATTERN 3: FEW-SHOT OUTPUT ===")
print(response_few.text.strip())
print(f"\nPrompt Tokens: {response_few.usage_metadata.prompt_token_count} | Candidate Tokens: {response_few.usage_metadata.candidates_token_count}")

=== PATTERN 3: FEW-SHOT OUTPUT ===
{
  "intent": "missing_items",
  "order_id": "74921",
  "units_missing": 2,
  "refund_requested_usd": 180.0,
  "urgency": "high"
}

Prompt Tokens: 294 | Candidate Tokens: 46


---
## Pattern 4: XML Structure & System Instruction (Production Standard)

In production architectures, we use two structural defenses:
1. **`system_instruction`:** Establishes the model's persona and invariants outside the user prompt.
2. **XML boundary tags:** Encloses context, rules, examples, and untrusted inputs in `<rules>`, `<examples>`, and `<input_ticket>` tags to prevent formatting corruption and instruction injection.

In [6]:
xml_prompt = f"""<rules>
1. Output valid JSON matching schema: {{"intent": string, "order_id": string|null, "units_missing": int, "refund_requested_usd": float|null, "urgency": string}}.
2. Allowed intents: missing_items, shipping_delay, product_inquiry, cancellation.
3. Allowed urgencies: high, medium, low.
</rules>

<examples>
<example>
<ticket>Where is order #92011? It was due 3 days ago.</ticket>
<output>{{"intent": "shipping_delay", "order_id": "92011", "units_missing": 0, "refund_requested_usd": null, "urgency": "medium"}}</output>
</example>
<example>
<ticket>Ordered 4 keyboards under order #55120. Box had only 2 keyboards. Refund $80 please.</ticket>
<output>{{"intent": "missing_items", "order_id": "55120", "units_missing": 2, "refund_requested_usd": 80.0, "urgency": "high"}}</output>
</example>
</examples>

<input_ticket>
{ANCHOR_TICKET}
</input_ticket>"""

config_xml = types.GenerateContentConfig(
    system_instruction="You are a production customer support data extraction pipeline. Output strictly valid JSON without preamble or explanations.",
    temperature=0.0
)

response_xml = client.models.generate_content(
    model='gemini-3.5-flash',
    contents=xml_prompt,
    config=config_xml
)

print("=== PATTERN 4: XML STRUCTURED OUTPUT ===")
print(response_xml.text.strip())
print(f"\nPrompt Tokens: {response_xml.usage_metadata.prompt_token_count} | Candidate Tokens: {response_xml.usage_metadata.candidates_token_count}")

=== PATTERN 4: XML STRUCTURED OUTPUT ===
{
  "intent": "missing_items",
  "order_id": "74921",
  "units_missing": 2,
  "refund_requested_usd": 180.0,
  "urgency": "high"
}

Prompt Tokens: 242 | Candidate Tokens: 44


---
## Summary: Empirical Pattern Comparison Table

Let us inspect the concrete trade-offs across all 4 patterns:

In [7]:
print(f"{'PATTERN':<25} | {'PROMPT TOKENS':<15} | {'OUTPUT TOKENS':<15} | {'SCHEMA STABILITY':<18}")
print("-" * 80)
print(f"{'1. Zero-Shot':<25} | {'92 tokens':<15} | {'48 tokens':<15} | {'Low (Drifts)':<18}")
print(f"{'2. One-Shot':<25} | {'168 tokens':<15} | {'46 tokens':<15} | {'Medium':<18}")
print(f"{'3. Few-Shot':<25} | {'294 tokens':<15} | {'46 tokens':<15} | {'High':<18}")
print(f"{'4. XML + System':<25} | {'242 tokens':<15} | {'44 tokens':<15} | {'Production Clean':<18}")
print("-" * 80)


PATTERN                   | PROMPT TOKENS   | OUTPUT TOKENS   | SCHEMA STABILITY  
--------------------------------------------------------------------------------
1. Zero-Shot              | 92 tokens       | 48 tokens       | Low (Drifts)      
2. One-Shot               | 168 tokens      | 46 tokens       | Medium            
3. Few-Shot               | 294 tokens      | 46 tokens       | High              
4. XML + System           | 242 tokens      | 44 tokens       | Production Clean  
--------------------------------------------------------------------------------

Key Systemic Takeaway:
- Zero-shot saves prompt tokens upfront but produces higher candidate token variance and requires parsing cleanup.
- XML demarcated Few-Shot invests more prompt tokens, but guarantees deterministic JSON payloads and minimizes output tokens.


---
## Step 5: Student Playground (Hands-On Exercise)

Now test your own ticket in `custom_ticket` below. Try ambiguous or challenging text (e.g. no order number, or an angry tone) and observe how the XML prompt pattern processes it.

In [8]:
# Enter your custom customer ticket here:
custom_ticket = """Hey team, my laptop charger stopped working after 2 days. Order number is unknown, bought it under email john@example.com. Send replacement ASAP."""

custom_prompt = f"""<rules>
1. Output valid JSON: {{"intent": string, "order_id": string|null, "units_missing": int, "refund_requested_usd": float|null, "urgency": string}}.
</rules>
<input_ticket>
{custom_ticket}
</input_ticket>"""

resp_custom = client.models.generate_content(
    model='gemini-3.5-flash',
    contents=custom_prompt,
    config=config_xml
)

print("Processed JSON:")
print(resp_custom.text.strip())
print(f"\n[TELEMETRY] Prompt tokens: {resp_custom.usage_metadata.prompt_token_count} | Candidate tokens: {resp_custom.usage_metadata.candidates_token_count}")

Processed JSON:
{
  "intent": "product_inquiry",
  "order_id": null,
  "units_missing": 0,
  "refund_requested_usd": null,
  "urgency": "high"
}

[TELEMETRY] Prompt tokens: 128 | Candidate tokens: 42
